# 02 - Silver : nettoyage des données Vélib'
**Objectif :** transformer les tables Bronze (brutes) en tables propres et fiables.

Deux tables :
- **`stations`** : une ligne par station (nom, position GPS, capacité), mise à jour avec `MERGE`
- **`station_status`** : un relevé par station et par heure (vélos mécaniques, électriques, places libres)

Nettoyages faits :
- noms de colonnes clairs et types corrects
- heure convertie en **heure de Paris**
- vélos mécaniques / électriques remis à plat
- stations non installées retirées
- pas de doublons


## 1. Table des stations
On crée la table si elle n'existe pas, puis on la met à jour avec `MERGE INTO` :
- station déjà connue et infos plus récentes → **mise à jour**
- nouvelle station → **ajout**

In [0]:
%sql
-- Structure de la table des stations (une ligne par station)
CREATE TABLE IF NOT EXISTS workspace.velib_silver.stations (
  station_id          BIGINT,
  station_code        STRING,
  name                STRING,
  latitude            DOUBLE,
  longitude           DOUBLE,
  capacity            INT,
  accepts_credit_card BOOLEAN,   -- paiement CB possible à la borne
  updated_at          TIMESTAMP
);

In [0]:
%sql
-- Mise à jour de la table Silver des stations à partir du Bronze
-- MERGE = on compare la table Silver (cible) avec les données Bronze (source)

MERGE INTO workspace.velib_silver.stations AS cible   -- table à mettre à jour

USING (
  -- Données sources : stations du Bronze, nettoyées
  SELECT
    station_id,                                                   -- identifiant unique de la station
    stationCode                                                  AS station_code,  -- code station (renommé)
    trim(name)                                                   AS name,          -- nom sans espaces en trop
    lat                                                          AS latitude,      -- renommé en clair
    lon                                                          AS longitude,     -- renommé en clair
    CAST(capacity AS INT)                                        AS capacity,      -- nombre de bornes, en entier
    coalesce(array_contains(rental_methods, 'CREDITCARD'), false) AS accepts_credit_card, -- true si paiement CB, false sinon (même si vide)
    ingestion_time                                               AS updated_at     -- date de chargement = date de mise à jour
  FROM workspace.velib_bronze.station_information
  -- Si une station apparaît dans plusieurs fichiers, on garde seulement la version la plus récente
  QUALIFY ROW_NUMBER() OVER (PARTITION BY station_id ORDER BY ingestion_time DESC) = 1
) AS source

-- Comment on reconnaît une même station dans les deux tables
ON cible.station_id = source.station_id

-- Cas 1 : la station existe déjà ET la source est plus récente → on met à jour toutes les colonnes
WHEN MATCHED AND source.updated_at > cible.updated_at THEN UPDATE SET *

-- Cas 2 : la station n'existe pas encore dans Silver → on l'ajoute
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1519,0,0,1519


## 2. Table des relevés de disponibilité
Un relevé = l'état d'une station à une heure donnée.
On ajoute seulement les relevés **pas encore présents** (clé : station + heure du relevé).

In [0]:
%sql
-- Structure de la table des relevés (1 relevé = 1 station à une heure donnée)
CREATE TABLE IF NOT EXISTS workspace.velib_silver.station_status (
  station_id         BIGINT,
  station_code       STRING,
  snapshot_time      TIMESTAMP,  -- heure du relevé (Paris)
  last_reported_time TIMESTAMP,  -- dernier signal envoyé par la station (Paris)
  bikes_available    INT,
  mechanical_bikes   INT,
  ebikes             INT,
  docks_available    INT,        -- places libres
  is_renting         BOOLEAN,    -- on peut prendre un vélo
  is_returning       BOOLEAN,    -- on peut rendre un vélo
  source_file        STRING,
  ingestion_time     TIMESTAMP
);

In [0]:
%sql
-- =====================================================================
-- Ajout des nouveaux relevés dans la table Silver
-- On prend les données du Bronze, on les nettoie,
-- puis on ajoute seulement celles qui ne sont pas encore dans Silver
-- =====================================================================

MERGE INTO workspace.velib_silver.station_status AS cible   -- table à remplir (Silver)

USING (
  -- ---------- Données sources : Bronze nettoyé ----------
  SELECT
    station_id,                          -- identifiant de la station (inchangé)
    stationCode AS station_code,         -- renommé en minuscules avec _

    -- Le Bronze donne l'heure en secondes depuis 1970 (ex : 1792860231)
    -- timestamp_seconds → transforme ce nombre en vraie date (en heure UTC)
    -- from_utc_timestamp → convertit l'heure UTC en heure de Paris
    from_utc_timestamp(timestamp_seconds(last_updated_epoch), 'Europe/Paris') AS snapshot_time,

    -- Dernier signal envoyé par la station (même conversion)
    -- Permet de repérer les stations déconnectées (données périmées)
    from_utc_timestamp(timestamp_seconds(last_reported), 'Europe/Paris') AS last_reported_time,

    -- CAST(... AS INT) → convertit la valeur en nombre entier
    CAST(num_bikes_available AS INT) AS bikes_available,

    -- La liste ressemble à : [{"ebike": null, "mechanical": 1}, {"ebike": 2, "mechanical": null}]
    -- aggregate → parcourt la liste et additionne les valeurs, en partant de 0
    -- coalesce(x.mechanical, 0) → remplace null par 0, pour pouvoir additionner
    CAST(aggregate(num_bikes_available_types, 0L, (total, x) -> total + coalesce(x.mechanical, 0)) AS INT) AS mechanical_bikes,  -- ici : 0 + 1 + 0 = 1
    CAST(aggregate(num_bikes_available_types, 0L, (total, x) -> total + coalesce(x.ebike, 0)) AS INT)      AS ebikes,            -- ici : 0 + 0 + 2 = 2

    CAST(num_docks_available AS INT) AS docks_available,   -- places libres, en entier

    -- Le Bronze utilise 1 / 0 → on transforme en true / false
    is_renting = 1   AS is_renting,      -- true si on peut prendre un vélo
    is_returning = 1 AS is_returning,    -- true si on peut rendre un vélo

    source_file,                         -- fichier d'origine (pour la traçabilité)
    ingestion_time                       -- date de chargement en Bronze

  FROM workspace.velib_bronze.station_status

  WHERE is_installed = 1   -- on garde seulement les stations installées

  -- Anti-doublons : si un même relevé (station + heure) apparaît plusieurs fois,
  -- on garde seulement le plus récemment chargé
  QUALIFY ROW_NUMBER() OVER (PARTITION BY station_id, last_updated_epoch ORDER BY ingestion_time DESC) = 1
) AS source

-- ---------- Comment reconnaître un relevé déjà présent ----------
-- Même station ET même heure de relevé = même relevé
ON  cible.station_id    = source.station_id
AND cible.snapshot_time = source.snapshot_time

-- ---------- Que faire ? ----------
-- Relevé pas encore dans Silver → on l'ajoute (toutes les colonnes)
-- Relevé déjà présent → on ne fait rien
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1516,0,0,1516


## 3. Contrôles qualité
On vérifie que les données Silver sont cohérentes :
- `incoherences_velos` : vélos dispos ≠ mécaniques + électriques → doit être à **0**
- `valeurs_negatives` : nombres négatifs → doit être à **0**

In [0]:
%sql
-- Contrôles qualité sur la table Silver des relevés
-- Objectif : vérifier que les données sont complètes et cohérentes

SELECT
  -- Nombre total de relevés (1 relevé = 1 station à une heure donnée)
  COUNT(*)                   AS nb_releves,

  -- Nombre de stations différentes (environ 1 500 attendues)
  COUNT(DISTINCT station_id) AS nb_stations,

  -- Lignes où vélos dispos ≠ mécaniques + électriques → doit être 0
  SUM(CASE WHEN bikes_available <> mechanical_bikes + ebikes THEN 1 ELSE 0 END) AS incoherences_velos,

  -- Lignes avec un nombre de vélos ou de places négatif (impossible) → doit être 0
  SUM(CASE WHEN bikes_available < 0 OR docks_available < 0 THEN 1 ELSE 0 END)   AS valeurs_negatives,

  -- Premier et dernier relevé, en heure de Paris
  MIN(snapshot_time)         AS premier_releve,
  MAX(snapshot_time)         AS dernier_releve

FROM workspace.velib_silver.station_status;

nb_releves,nb_stations,incoherences_velos,valeurs_negatives,premier_releve,dernier_releve
1516,1516,0,0,2026-10-24T18:50:58.000Z,2026-10-24T18:50:58.000Z


In [0]:
%sql
-- Aperçu : relevés avec le nom de la station
SELECT s.name, r.snapshot_time, r.mechanical_bikes, r.ebikes, r.docks_available, s.capacity
FROM workspace.velib_silver.station_status AS r
JOIN workspace.velib_silver.stations      AS s ON r.station_id = s.station_id
ORDER BY s.name
LIMIT 20;

name,snapshot_time,mechanical_bikes,ebikes,docks_available,capacity
11 Novembre 1918 - 8 Mai 1945,2026-10-24T18:50:58.000Z,2,27,5,36
18 juin 1940 - Buzenval,2026-10-24T18:50:58.000Z,0,4,21,25
8 Mai 1945 - 10 Juillet 1940,2026-10-24T18:50:58.000Z,14,15,1,30
Abbeville - Faubourg Poissonnière,2026-10-24T18:50:58.000Z,1,2,10,14
Abbé Carton - Plantes,2026-10-24T18:50:58.000Z,1,1,22,25
Abbé Groult - Convention,2026-10-24T18:50:58.000Z,0,1,25,27
Aboukir - Alexandrie,2026-10-24T18:50:58.000Z,1,1,17,21
Adolphe Lalyre - Armand Silvestre,2026-10-24T18:50:58.000Z,6,4,30,40
Adolphe Pinard - Victor Hugo,2026-10-24T18:50:58.000Z,3,1,47,52
Adrien Lesesne - Docteur Bauer,2026-10-24T18:50:58.000Z,5,1,23,30


## Remarques
- 3 stations retirées car non installées (1 519 en Bronze → 1 516 en Silver)
- `snapshot_time` est en **heure de Paris**, même si Databricks affiche `+00:00` (sa session est en UTC)
- Relancer le MERGE n'ajoute pas de doublons